# validation-no-grad — worked example 3: Validation via @torch.no_grad decorator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `validation-no-grad`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`@torch.no_grad()` as a function decorator disables autograd for the whole function body — the cleaner equivalent of wrapping everything in a `with` block. Pairing it with `model.eval()` is the canonical eval setup: dropout off, BN stats frozen, and no graph built.

## Worked solution

We write a decorated validation routine and prove autograd is off inside it.

1. We decorate `validate` with `@torch.no_grad()`; every tensor created in the body is graph-free.
2. Inside, we call `model.eval()` to flip eval-mode layers, forward the model, take predictions, and compute accuracy.
3. We read `torch.is_grad_enabled()` inside the function — it returns False, the proof that the decorator is in effect even though the caller has grad enabled.

We print the accuracy and the `grad_enabled_inside` flag.

In [ ]:
import torch
import torch as t
import torch.nn as nn

t.manual_seed(2)
model = nn.Linear(6, 4)
x = t.randn(5, 6)
y = t.randint(0, 4, (5,))

@torch.no_grad()
def validate(model, x, y):
    model.eval()
    logits = model(x)
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean().item()
    return {'acc': float(acc), 'grad_enabled_inside': bool(torch.is_grad_enabled())}

out = validate(model, x, y)
print('acc:', round(out['acc'], 3))
print('grad_enabled_inside:', out['grad_enabled_inside'])